# Quiz 2: Dataset Generation
**Name:** Francis Budu Gyampo PUIT/232210108

This notebook generates a synthetic dataset of 5000 records. Run all cells to view it.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
rng = np.random.default_rng(42)  # fixed seed, so the same data appears on every run
n = 5000

departments = ["IT", "Business", "Nursing", "Education", "Engineering"]

df = pd.DataFrame({
    "student_id": np.arange(1, n + 1),
    "age": rng.integers(17, 30, n),
    "gender": rng.choice(["Male", "Female"], n),
    "department": rng.choice(departments, n),
    "study_hours_per_week": rng.integers(1, 41, n),
    "attendance_rate": rng.integers(50, 101, n),
})

# scores depend a little on study hours and attendance, plus some randomness
df["quiz_score"] = (
    30 + df["study_hours_per_week"] * 0.8 + df["attendance_rate"] * 0.3
    + rng.normal(0, 8, n)
).clip(0, 100).round(1)

df["final_score"] = (
    df["quiz_score"] * 0.6 + rng.normal(20, 8, n)
).clip(0, 100).round(1)

df["passed"] = np.where(df["final_score"] >= 50, "Yes", "No")

In [ ]:
print(df.shape)   # should show (5000, 9)
df.head(10)

In [ ]:
df.describe()

In [ ]:
df.to_csv("dataset.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, confusion_matrix, classification_report

## 1. Simple Linear Regression
Predicting final_score from quiz_score (one predictor).

In [ ]:
X = df[["quiz_score"]]
y = df["final_score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

slr = LinearRegression()
slr.fit(X_train, y_train)
pred = slr.predict(X_test)

print("Intercept:", round(slr.intercept_, 3))
print("Slope:", round(slr.coef_[0], 3))
print("R-squared:", round(r2_score(y_test, pred), 3))
print("RMSE:", round(mean_squared_error(y_test, pred) ** 0.5, 3))

plt.scatter(X_test, y_test, alpha=0.3, label="Actual")
plt.plot(X_test.sort_values("quiz_score"), slr.predict(X_test.sort_values("quiz_score")), color="red", label="Fitted line")
plt.xlabel("Quiz score")
plt.ylabel("Final score")
plt.title("Simple Linear Regression")
plt.legend()
plt.show()

In [ ]:
features = ["quiz_score", "study_hours_per_week", "attendance_rate", "age"]
X = df[features]
y = df["final_score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

mlr = LinearRegression()
mlr.fit(X_train, y_train)
pred = mlr.predict(X_test)

print("Intercept:", round(mlr.intercept_, 3))
for name, coef in zip(features, mlr.coef_):
    print(f"{name}: {coef:.3f}")
print("R-squared:", round(r2_score(y_test, pred), 3))
print("RMSE:", round(mean_squared_error(y_test, pred) ** 0.5, 3))

## 3. Logistic Regression
Predicting whether a student passed (Yes/No).

In [ ]:
features = ["quiz_score", "study_hours_per_week", "attendance_rate"]
X = df[features]
y = (df["passed"] == "Yes").astype(int)   # Yes = 1, No = 0

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

logr = LogisticRegression(max_iter=1000)
logr.fit(X_train, y_train)
pred = logr.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("Confusion matrix:")
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))